In [1]:
# ============================================================
# INDIAN QUANT PORTFOLIO & RISK ENGINE
# Data Ingestion Pipeline
# ============================================================

import os
import time
import yfinance as yf
import pandas as pd
import numpy as np


# ============================================================
# CONFIGURATION
# ============================================================

START_DATE = "2015-01-01"
END_DATE = None          # None = up to latest available date
INTERVAL = "1d"

OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# INDIAN EQUITY UNIVERSE
# ============================================================
# Keep the initial universe reasonably diversified.
# We can later expand this to 50-100 stocks.

STOCKS = {

    # Financials
    "HDFCBANK.NS": "HDFC Bank",
    "ICICIBANK.NS": "ICICI Bank",
    "SBIN.NS": "State Bank of India",
    "AXISBANK.NS": "Axis Bank",
    "KOTAKBANK.NS": "Kotak Mahindra Bank",

    # IT
    "TCS.NS": "TCS",
    "INFY.NS": "Infosys",
    "HCLTECH.NS": "HCL Technologies",
    "WIPRO.NS": "Wipro",
    "TECHM.NS": "Tech Mahindra",

    # Energy
    "RELIANCE.NS": "Reliance Industries",
    "ONGC.NS": "ONGC",
    "NTPC.NS": "NTPC",
    "POWERGRID.NS": "Power Grid",

    # FMCG
    "HINDUNILVR.NS": "Hindustan Unilever",
    "ITC.NS": "ITC",
    "NESTLEIND.NS": "Nestle India",
    "BRITANNIA.NS": "Britannia",

    # Automobile
    "MARUTI.NS": "Maruti Suzuki",
    "M&M.NS": "Mahindra & Mahindra",
    "TATAMOTORS.NS": "Tata Motors",
    "EICHERMOT.NS": "Eicher Motors",

    # Pharma / Healthcare
    "SUNPHARMA.NS": "Sun Pharma",
    "CIPLA.NS": "Cipla",
    "DRREDDY.NS": "Dr Reddy's",
    "APOLLOHOSP.NS": "Apollo Hospitals",

    # Telecom
    "BHARTIARTL.NS": "Bharti Airtel",

    # Metals
    "TATASTEEL.NS": "Tata Steel",
    "HINDALCO.NS": "Hindalco",
    "JSWSTEEL.NS": "JSW Steel",

    # Consumer / Retail
    "TITAN.NS": "Titan",
    "ASIANPAINT.NS": "Asian Paints",

    # Industrials
    "LT.NS": "Larsen & Toubro",
    "ADANIPORTS.NS": "Adani Ports",

    # Real Estate
    "DLF.NS": "DLF",
}


# ============================================================
# BENCHMARK + MARKET RISK
# ============================================================

MARKET_TICKERS = {

    "^NSEI": "NIFTY 50",

    "^INDIAVIX": "India VIX",
}


# ============================================================
# DOWNLOAD FUNCTION
# ============================================================

def download_data(tickers, start_date, end_date=None):

    print("\n" + "=" * 70)
    print("Downloading market data...")
    print("=" * 70)

    print(f"Number of instruments: {len(tickers)}")
    print(f"Start date: {start_date}")
    print(f"End date: {end_date}")
    print(f"Interval: {INTERVAL}")

    data = yf.download(
        tickers=list(tickers),
        start=start_date,
        end=end_date,
        interval=INTERVAL,

        # Important for portfolio analysis
        auto_adjust=True,

        # Organize data by ticker
        group_by="ticker",

        # Faster for multiple stocks
        threads=True,

        # Don't include pre/post market
        prepost=False,

        # Try to repair obvious Yahoo data inconsistencies
        repair=True,

        progress=True
    )

    return data


# ============================================================
# DOWNLOAD EQUITIES
# ============================================================

equity_data = download_data(
    STOCKS.keys(),
    START_DATE,
    END_DATE
)


# ============================================================
# CHECK DOWNLOAD
# ============================================================

if equity_data is None or equity_data.empty:
    raise RuntimeError("No equity data was downloaded.")


print("\nDownloaded equity data:")
print(equity_data.shape)

print("\nAvailable tickers:")
print(equity_data.columns.get_level_values(0).unique().tolist())


# ============================================================
# SAVE RAW EQUITY DATA
# ============================================================

equity_raw_path = os.path.join(
    OUTPUT_DIR,
    "indian_equities_raw.csv"
)

equity_data.to_csv(equity_raw_path)

print(f"\nSaved raw equity data to:")
print(equity_raw_path)


# ============================================================
# EXTRACT ADJUSTED CLOSE / CLOSE
# ============================================================
# Since auto_adjust=True, the "Close" series is already
# adjusted for corporate actions according to yfinance.

prices = pd.DataFrame(index=equity_data.index)

failed_tickers = []

for ticker in STOCKS.keys():

    try:

        if ticker in equity_data.columns.get_level_values(0):

            prices[ticker] = equity_data[ticker]["Close"]

        else:

            failed_tickers.append(ticker)

    except Exception as e:

        print(f"Could not process {ticker}: {e}")
        failed_tickers.append(ticker)


# ============================================================
# CLEAN PRICE DATA
# ============================================================

prices.index.name = "Date"

# Sort chronologically
prices = prices.sort_index()

# Remove completely empty columns
prices = prices.dropna(axis=1, how="all")

# Forward-fill only short gaps
prices = prices.ffill(limit=5)


# ============================================================
# REMOVE STOCKS WITH EXCESSIVE MISSING DATA
# ============================================================

missing_ratio = prices.isna().mean()

MAX_MISSING_RATIO = 0.05

bad_stocks = missing_ratio[
    missing_ratio > MAX_MISSING_RATIO
].index.tolist()

if bad_stocks:

    print("\nRemoving stocks with excessive missing data:")

    for stock in bad_stocks:
        print(
            f"{stock}: "
            f"{missing_ratio[stock] * 100:.2f}% missing"
        )

    prices = prices.drop(columns=bad_stocks)


# ============================================================
# RETURNS
# ============================================================

daily_returns = prices.pct_change(fill_method=None)

daily_returns = daily_returns.replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# BASIC DATA QUALITY CHECK
# ============================================================

print("\n" + "=" * 70)
print("DATA QUALITY CHECK")
print("=" * 70)

print("\nPrice data shape:")
print(prices.shape)

print("\nReturn data shape:")
print(daily_returns.shape)

print("\nMissing values:")
print(prices.isna().sum().sort_values(ascending=False).head(10))

print("\nDate range:")
print(prices.index.min(), "→", prices.index.max())


# ============================================================
# DOWNLOAD MARKET DATA
# ============================================================

market_data = download_data(
    MARKET_TICKERS.keys(),
    START_DATE,
    END_DATE
)


# ============================================================
# EXTRACT NIFTY + INDIA VIX
# ============================================================

market_prices = pd.DataFrame(index=market_data.index)

for ticker in MARKET_TICKERS.keys():

    try:

        market_prices[ticker] = market_data[ticker]["Close"]

    except Exception as e:

        print(f"Could not process market ticker {ticker}: {e}")


market_prices.index.name = "Date"

market_prices = market_prices.sort_index()

market_prices = market_prices.ffill(limit=5)


# ============================================================
# SAVE MARKET DATA
# ============================================================

market_raw_path = os.path.join(
    OUTPUT_DIR,
    "indian_market_raw.csv"
)

market_data.to_csv(market_raw_path)

market_price_path = os.path.join(
    OUTPUT_DIR,
    "market_prices.csv"
)

market_prices.to_csv(market_price_path)


# ============================================================
# SAVE CLEAN DATASETS
# ============================================================

prices_path = os.path.join(
    OUTPUT_DIR,
    "adjusted_close_prices.csv"
)

returns_path = os.path.join(
    OUTPUT_DIR,
    "daily_returns.csv"
)

prices.to_csv(prices_path)

daily_returns.to_csv(returns_path)


# ============================================================
# CREATE MASTER DATASET
# ============================================================

master_data = prices.copy()

master_data.columns = [
    f"{ticker}_PRICE"
    for ticker in master_data.columns
]

for ticker in daily_returns.columns:

    master_data[
        f"{ticker}_RETURN"
    ] = daily_returns[ticker]


# Add market variables
for ticker in market_prices.columns:

    master_data[
        f"{ticker}_MARKET"
    ] = market_prices[ticker]


# ============================================================
# SAVE MASTER DATASET
# ============================================================

master_path = os.path.join(
    OUTPUT_DIR,
    "master_market_dataset.csv"
)

master_data.to_csv(master_path)


# ============================================================
# DATA SUMMARY
# ============================================================

summary = pd.DataFrame({

    "Ticker": prices.columns,

    "Start_Date": [
        prices[col].first_valid_index()
        for col in prices.columns
    ],

    "End_Date": [
        prices[col].last_valid_index()
        for col in prices.columns
    ],

    "Observations": [
        prices[col].notna().sum()
        for col in prices.columns
    ],

    "Missing_%": [
        prices[col].isna().mean() * 100
        for col in prices.columns
    ],

    "Annualized_Return_%": [
        daily_returns[col].mean() * 252 * 100
        for col in prices.columns
    ],

    "Annualized_Volatility_%": [
        daily_returns[col].std() * np.sqrt(252) * 100
        for col in prices.columns
    ]

})


summary_path = os.path.join(
    OUTPUT_DIR,
    "data_summary.csv"
)

summary.to_csv(summary_path, index=False)


# ============================================================
# FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("DATA PIPELINE COMPLETED")
print("=" * 70)

print(f"\nStocks requested: {len(STOCKS)}")
print(f"Stocks downloaded: {len(prices.columns)}")
print(f"Stocks failed/removed: {len(failed_tickers) + len(bad_stocks)}")

if failed_tickers:
    print("\nFailed tickers:")
    print(failed_tickers)

print("\nFiles created:")

print(f"1. {equity_raw_path}")
print(f"2. {market_raw_path}")
print(f"3. {market_price_path}")
print(f"4. {prices_path}")
print(f"5. {returns_path}")
print(f"6. {master_path}")
print(f"7. {summary_path}")

print("\n" + "=" * 70)
print("Ready for portfolio optimization.")
print("=" * 70)


Number of instruments: 35
Start date: 2015-01-01
End date: None
Interval: 1d


Failed to get ticker 'TATAMOTORS.NS' reason: Failed to perform, curl: (28) Connection timed out after 30001 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$HINDALCO.NS: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-08-20)
[****                   9%                       ]  3 of 35 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$POWERGRID.NS: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-08-20)
[*****                 11%                       ]  4 of 35 completed$ICICIBANK.NS: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-08-20)
$HINDUNILVR.NS: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-08-20)
[*******               14%                       ]  5 of 35 completed$MARUTI.NS: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-08-20)
[********              17%             


Downloaded equity data:
(2875, 210)

Available tickers:
['HINDALCO.NS', 'SUNPHARMA.NS', 'TATASTEEL.NS', 'POWERGRID.NS', 'ICICIBANK.NS', 'HINDUNILVR.NS', 'MARUTI.NS', 'ADANIPORTS.NS', 'DRREDDY.NS', 'KOTAKBANK.NS', 'DLF.NS', 'TATAMOTORS.NS', 'EICHERMOT.NS', 'AXISBANK.NS', 'SBIN.NS', 'JSWSTEEL.NS', 'RELIANCE.NS', 'HDFCBANK.NS', 'TITAN.NS', 'HCLTECH.NS', 'NESTLEIND.NS', 'M&M.NS', 'LT.NS', 'TCS.NS', 'ASIANPAINT.NS', 'CIPLA.NS', 'ITC.NS', 'APOLLOHOSP.NS', 'WIPRO.NS', 'BRITANNIA.NS', 'TECHM.NS', 'ONGC.NS', 'INFY.NS', 'BHARTIARTL.NS', 'NTPC.NS']

Saved raw equity data to:
data/raw/indian_equities_raw.csv

DATA QUALITY CHECK

Price data shape:
(2875, 15)

Return data shape:
(2875, 15)

Missing values:
KOTAKBANK.NS    0
INFY.NS         0
WIPRO.NS        0
TECHM.NS        0
ONGC.NS         0
NTPC.NS         0
ITC.NS          0
NESTLEIND.NS    0
BRITANNIA.NS    0
SUNPHARMA.NS    0
dtype: int64

Date range:
2015-01-01 00:00:00 → 2026-08-19 00:00:00

Number of instruments: 2
Start date: 2015-01-01


[*********************100%***********************]  2 of 2 completed


DATA PIPELINE COMPLETED

Stocks requested: 35
Stocks downloaded: 15
Stocks failed/removed: 0

Files created:
1. data/raw/indian_equities_raw.csv
2. data/raw/indian_market_raw.csv
3. data/raw/market_prices.csv
4. data/raw/adjusted_close_prices.csv
5. data/raw/daily_returns.csv
6. data/raw/master_market_dataset.csv
7. data/raw/data_summary.csv

Ready for portfolio optimization.
